# 04 — Bonus Analyses

Additional analyses that demonstrate depth of understanding:
1. Tokenization deep dive — how Llama-3's BPE fragments Turkish morphology
2. English pivot detection — replicating Wendler et al. (2024) on reasoning tasks
3. Difficulty interaction — do harder problems diverge earlier?
4. Attention pattern analysis (if time permits)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.utils import load_cached, set_seed
from src.model import LlamaWrapper
from src.visualization import set_publication_style, COLOR_EN, COLOR_TR

set_seed(42)
set_publication_style()

data = load_cached('../results/cached_activations/analysis_results.pt')
problems = data['problems']

## 1. Tokenization Deep Dive

Turkish agglutinative morphology means single words carry what English needs a phrase for. When the tokenizer (trained mostly on English) splits these, it often breaks at morpheme boundaries incorrectly.

In [ ]:
# Load model for tokenization analysis
wrapper = LlamaWrapper(
    model_name="meta-llama/Meta-Llama-3-8B",
    dtype="float16",
)

# Turkish morphology examples
examples = [
    ("evlerinizdekilerden", "from those in your houses", "ev-ler-iniz-deki-ler-den"),
    ("yapabileceklerimizden", "from the things we can do", "yap-abil-ecek-ler-imiz-den"),
    ("hesaplayamayacaklarını", "that they won't be able to calculate", "hesapla-y-ama-y-acak-ları-nı"),
    ("topluyordu", "was collecting", "topl-u-yor-du"),
    ("kazandığından", "because he/she earned", "kazan-dığ-ı-n-dan"),
]

print("Turkish Morphology vs Llama-3 Tokenization:\n")
for word, english, morphemes in examples:
    tokens = wrapper.tokenizer.encode(word, add_special_tokens=False)
    decoded = [wrapper.tokenizer.decode([t]) for t in tokens]
    print(f"  Word: {word}")
    print(f"  Meaning: {english}")
    print(f"  Morphemes: {morphemes}")
    print(f"  BPE tokens: {decoded} ({len(tokens)} tokens)")
    
    # Compare with English
    en_tokens = wrapper.tokenizer.encode(english, add_special_tokens=False)
    print(f"  English tokens: {len(en_tokens)} tokens")
    print()

In [ ]:
# Tokens per word distribution across all 30 problems
fertility_data = pd.read_csv('../results/tables/fertility_analysis.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Tokens per word
ax = axes[0]
ax.hist(fertility_data['tokens_per_word_en'], bins=15, alpha=0.7, color=COLOR_EN, label='English')
ax.hist(fertility_data['tokens_per_word_tr'], bins=15, alpha=0.7, color=COLOR_TR, label='Turkish')
ax.set_xlabel('Tokens per Word')
ax.set_ylabel('Count')
ax.set_title('Tokenization Density')
ax.legend()

# Fertility ratio distribution
ax = axes[1]
ax.hist(fertility_data['fertility_ratio'], bins=15, color='#6366F1', alpha=0.7)
ax.axvline(1.0, color='gray', linestyle='--', label='Equal (1.0x)')
ax.axvline(fertility_data['fertility_ratio'].mean(), color='red', linestyle='--',
           label=f"Mean ({fertility_data['fertility_ratio'].mean():.2f}x)")
ax.set_xlabel('Fertility Ratio (TR/EN tokens)')
ax.set_ylabel('Count')
ax.set_title('Tokenization Tax on Turkish')
ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/fig_bonus_tokenization.pdf', dpi=300)
plt.savefig('../results/figures/fig_bonus_tokenization.png', dpi=300)
plt.show()

## 2. English Pivot Detection

Wendler et al. (2024) showed LLMs reason in English internally even for non-English inputs. We test this specifically for mathematical reasoning: at each layer, when the input is Turkish, is the model's top prediction an English token?

In [ ]:
# For each layer, count how many Turkish prompts produce ASCII (likely English) top-1 predictions
n_layers = len(data['all_results_tr'][0]['top_tokens'])
n_problems = len(data['all_results_tr'])

ascii_fraction = np.zeros(n_layers)
for layer_idx in range(n_layers):
    ascii_count = 0
    for result in data['all_results_tr']:
        if result['top_tokens'][layer_idx]:
            token_str = result['top_tokens'][layer_idx][0][1].strip()
            if all(ord(c) < 128 for c in token_str) and token_str:
                ascii_count += 1
    ascii_fraction[layer_idx] = ascii_count / n_problems

fig, ax = plt.subplots(figsize=(10, 5))
layers = np.arange(n_layers)
ax.bar(layers, ascii_fraction * 100, color='#6366F1', alpha=0.8)
ax.axhspan(0, 50, alpha=0.05, color='red')
ax.axhspan(50, 100, alpha=0.05, color='blue')
ax.set_xlabel('Layer')
ax.set_ylabel('% Turkish prompts with ASCII top-1 prediction')
ax.set_title('English Pivot Detection for Turkish Math Prompts\n(Wendler et al. 2024 replication on reasoning tasks)')
ax.set_ylim(0, 105)

# Annotate
mid_start, mid_end = 11, 21
mid_mean = ascii_fraction[mid_start:mid_end+1].mean() * 100
ax.annotate(f'Mid-layer mean: {mid_mean:.0f}%',
            xy=((mid_start + mid_end) / 2, mid_mean + 5),
            fontsize=10, ha='center', color='red')

plt.tight_layout()
plt.savefig('../results/figures/fig_bonus_english_pivot.pdf', dpi=300)
plt.savefig('../results/figures/fig_bonus_english_pivot.png', dpi=300)
plt.show()

print(f"\nIf mid-layers show high ASCII fraction, the model pivots through English for Turkish math reasoning.")

## 3. Difficulty Interaction

Do harder problems (more reasoning steps) show earlier divergence between English and Turkish?

In [ ]:
# Scatter: n_steps vs divergence onset layer
valid_data = [(p['n_steps'], onset) for p, onset in zip(problems, data['onset_layers']) if onset >= 0]

if valid_data:
    steps, onsets = zip(*valid_data)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(steps, onsets, s=80, alpha=0.6, c='#6366F1', edgecolors='white')
    
    # Trend line
    z = np.polyfit(steps, onsets, 1)
    p_trend = np.poly1d(z)
    x_line = np.linspace(min(steps), max(steps), 100)
    ax.plot(x_line, p_trend(x_line), '--', color='red', alpha=0.7, label=f'Trend (slope={z[0]:.2f})')
    
    ax.set_xlabel('Number of Reasoning Steps')
    ax.set_ylabel('Divergence Onset Layer')
    ax.set_title('Do Harder Problems Diverge Earlier?')
    ax.legend()
    
    # Correlation
    from scipy.stats import spearmanr
    rho, p_val = spearmanr(steps, onsets)
    ax.text(0.05, 0.95, f'Spearman ρ = {rho:.2f} (p = {p_val:.3f})',
            transform=ax.transAxes, fontsize=10, va='top')
    
    plt.tight_layout()
    plt.savefig('../results/figures/fig_bonus_difficulty_interaction.pdf', dpi=300)
    plt.savefig('../results/figures/fig_bonus_difficulty_interaction.png', dpi=300)
    plt.show()
    
    if z[0] < 0:
        print("Negative slope: harder problems diverge EARLIER → supports hypothesis.")
    else:
        print("Positive/flat slope: difficulty doesn't affect WHERE divergence occurs.")
else:
    print("No divergence detected in any problem.")

In [ ]:
# Summary of bonus findings
print("="*60)
print("BONUS ANALYSES SUMMARY")
print("="*60)
print(f"\n1. Tokenization Tax:")
print(f"   Turkish uses ~{fertility_data['fertility_ratio'].mean():.1f}x more tokens than English")
print(f"   Turkish: {fertility_data['tokens_per_word_tr'].mean():.1f} tokens/word vs English: {fertility_data['tokens_per_word_en'].mean():.1f} tokens/word")
print(f"\n2. English Pivot:")
print(f"   Mid-layer (11-21) ASCII fraction: {ascii_fraction[11:22].mean()*100:.0f}%")
print(f"   Early (0-10): {ascii_fraction[:11].mean()*100:.0f}%, Late (22-32): {ascii_fraction[22:].mean()*100:.0f}%")
if valid_data:
    print(f"\n3. Difficulty Interaction:")
    print(f"   Spearman ρ = {rho:.2f} (p = {p_val:.3f})")